# 07 资源锁：时空占用表


本课按百科条目写。占用表 **不是** 搜索算法，但是时间 A\* / SIPP / HCA\* / CBS 底层的共同输入。调度台可跳过。

---

## 1. 定义

**占用表 / 预约表**（reservation table）记录「谁在何时占用哪」。多车、动态障碍、已下发路段，都先写成表上的键。

三类键：

| 锁 | 键 | 禁止 |
|----|----|------|
| 顶点 | \((v,t)\) | 两智能体同一时刻同一点 |
| 边 | \((u,v,t)\) | t 拍从 u 走到 v；对向 \((v,u,t)\) 同时占用 |
| 段 issued | \(v\) | 已承诺给某车、尚未走完的点（不带时间，执行层互斥） |

输入：占/释放请求。输出：成功或失败；查询 `vertex_free` / `edge_free`；SIPP 还要 `safe_intervals`。

时间离散成整数拍。一步移动或等待都消耗 1 拍（本课约定，与 08 一致）。

---

## 2. 和「空间最短路」的关系

普通 A\* 的状态是点。加上占用表之后，同一点在 t=1 空闲、t=2 被占，必须当成不同资源。下一课把时间写进状态；本课先把表写对。

CBS 的「约束」底层也是往这张表里写 block。HCA\* 则是高优先级车的整条时空轨迹。

---

## 3. 不变量

- `occupy` 失败 ⇒ **表不变**（不覆盖别人）。
- 自己占的格，`vertex_free` 对本人仍为真（允许在自己的预约上等待）。
- 边锁同时挡 \((a,b,t)\) 与 \((b,a,t)\)，防止对向换位擦肩。
- `issue_segment` 要么整段成功要么整段失败（不能锁一半）。
- `safe_intervals(v, T, agent)` 返回半开区间 \([lo,hi)\)，别人占用的整数拍不落在任何区间里。

---

## 4. 伪代码

```
occupy(v, t, agent):
    if table[v,t] 存在且 ≠ agent: return false
    table[v,t] ← agent; return true

vertex_free(v, t, agent):
    return table[v,t] 为空 或 = agent

occupy_edge(a, b, t, agent):
    若 (a,b,t) 或 (b,a,t) 被别人占: return false
    两向都记下 agent（或至少挡对向查询）

issue_segment(nodes, agent):
    若任一 node 已被别人 issued: return false
    全部标成 agent

safe_intervals(v, horizon, agent):
    blocked ← 别人占用 v 的时刻，排序，且 < horizon
    从 0 起，每个 blocked 拍把区间切开
    区间均为 [lo, hi) 半开
```

---

## 5. 手算

走廊点 C2。horizon=8。无人占用：一个区间 `[0, 8)`。

占 t=2（别人）：切开成 `[0, 2)` 和 `[3, 8)`。注意 **t=2 本身不在任何空闲区间**。SIPP 若误用闭区间，会在占用拍插入。

车路径 A,C1,C2 从 t=0 每步一格，顶点键：`(A,0),(C1,1),(C2,2)`。另一车从 B 对向走，禁止与这些键冲突，也禁止在相应拍走对向边。


In [ ]:
import sys
from pathlib import Path
ROOT = Path.cwd() if (Path.cwd() / "lib").exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT))


## 6. 自己写最小表：顶点 + 边 + 区间


In [ ]:
class MiniTable:
    def __init__(self):
        self.vertex = {}
        self.edge = {}
        self.issued_by = {}

    def occupy(self, node, t, agent):
        key = (node, t)
        holder = self.vertex.get(key)
        if holder is not None and holder != agent:
            return False
        self.vertex[key] = agent
        return True

    def free(self, node, t, agent):
        if self.vertex.get((node, t)) == agent:
            del self.vertex[(node, t)]

    def vertex_free(self, node, t, agent):
        holder = self.vertex.get((node, t))
        return holder is None or holder == agent

    def occupy_edge(self, a, b, t, agent):
        if self.edge.get((a, b, t)) not in (None, agent):
            return False
        if self.edge.get((b, a, t)) not in (None, agent):
            return False
        self.edge[(a, b, t)] = agent
        self.edge[(b, a, t)] = agent
        return True

    def edge_free(self, a, b, t, agent):
        return self.edge.get((a, b, t)) in (None, agent) and self.edge.get((b, a, t)) in (None, agent)

    def issue_segment(self, nodes, agent):
        for n in nodes:
            holder = self.issued_by.get(n)
            if holder is not None and holder != agent:
                return False
        for n in nodes:
            self.issued_by[n] = agent
        return True

    def release_segment(self, nodes, agent):
        for n in nodes:
            if self.issued_by.get(n) == agent:
                del self.issued_by[n]

    def safe_intervals(self, node, horizon, agent):
        blocked = sorted(
            t for (n, t), who in self.vertex.items()
            if n == node and who != agent and t < horizon
        )
        intervals = []
        start = 0
        for t in blocked:
            if t > start:
                intervals.append((start, t))
            start = t + 1
        if start < horizon:
            intervals.append((start, horizon))
        return intervals


table = MiniTable()
print("a 占 C2@t=1", table.occupy("C2", 1, "a"))
print("b 同键应失败", table.occupy("C2", 1, "b"))
print("b 占 t=2", table.occupy("C2", 2, "b"))
print("a 在自己的 t=1 仍 free", table.vertex_free("C2", 1, "a"))
print("b 在 t=1 不 free", table.vertex_free("C2", 1, "b"))
table.free("C2", 1, "a")
print("释放后 b 占 t=1", table.occupy("C2", 1, "b"))
print("空闲区间 horizon=8", table.safe_intervals("C2", 8, "me"))


对照手算：别人占了 t=1 和 t=2 之后，`me` 的区间应是 `[0,1)` 与 `[3,8)`。

## 7. 逐行

| 行 | 作用 |
|----|------|
| `key=(node,t)` | 空间×时间，缺一不可 |
| 先 get 再写 | 失败不覆盖 |
| 本人可再 occupy | 允许在自己的预约上等待 |
| 边写两向 | `edge_free` 对向也挡 |
| `issue` 先全检查再全写 | 禁止锁一半 |
| 区间半开、blocked 拍跳过 | 给 SIPP 用 |

库 `ReservationTable` 字段名相同：`vertex` / `edge` / `issued_by`。


In [ ]:
from lib.reservation import ReservationTable

t = ReservationTable()
print("a 下发 P0-P2", t.issue_segment(["P0", "P1", "P2"], "a"))
print("b 抢 P2", t.issue_segment(["P2", "P3"], "b"))
t.release_segment(["P0", "P1", "P2"], "a")
print("释放后 b", t.issue_segment(["P2", "P3"], "b"))

t2 = ReservationTable()
print("空表 C2", t2.safe_intervals("C2", 8, "me"))
t2.occupy("C2", 2, "x")
print("占 t=2 后", t2.safe_intervals("C2", 8, "me"))

print("边：a 占 A->C1 @t=0", t2.occupy_edge("A", "C1", 0, "a"))
print("对向 C1->A 同时", t2.occupy_edge("C1", "A", 0, "b"))
print("边空闲查询对向", t2.edge_free("C1", "A", 0, "b"))


## 常见 bug

1. 只锁点不锁边：对向在同一拍互换格子，顶点冲突检测在「各走各的下一格」时可能漏（取决于你是否先移动）。边锁一次挡住。
2. 解锁时 agent 名写错，把别人的锁删了。
3. `safe_intervals` 用闭区间，占用拍被当成空闲。
4. issued 锁忘了释放，后面的车永远过不了路口。
5. 时间键用了「到达拍」还是「离开拍」和 08 课边检查不一致。本课约定：`occupy_edge(u,v,t)` 表示 **t 拍正在从 u 走向 v**，下一课 `edge_free(u,v,t)` 用离开的那一拍。

## 条目小结

| 项目 | 内容 |
|------|------|
| 顶点键 | (点, 时刻) |
| 边键 | (u,v,t) 及对向 |
| 段锁 | 执行层互斥，不带时间 |
| 区间 | [lo,hi) 半开 |
| 下一课 | 状态改成 (点,时刻)，查这张表 |

## 练习

1. 车路径 A,C1,C2 从 t=0 每步一格，列出全部顶点键和边键。
2. 占用 {1,2,5}、horizon=8，手写 `safe_intervals`（agent 是别人）。
3. 为什么 `occupy` 失败必须整表不变？举一个「写了一半」导致死锁的例子。
